# Exercise 05 — BCM Plasticity on GPU

## Background

**Bienenstock-Cooper-Munro (BCM)** plasticity is a rate-based learning rule with a **sliding threshold** $\theta$.

For a synapse from pre-neuron $i$ to post-neuron $j$, the weight update is:

$$\frac{dw_{ij}}{dt} = \eta \cdot r_i \cdot r_j \cdot (r_j - \theta_j)$$

where:
- $r_i$, $r_j$ are smoothed firing rates (low-pass filtered spike trains)
- $\theta_j$ is the **modification threshold** — the rate at which the synapse switches from LTD to LTP
- $\theta$ slides: $\frac{d\theta_j}{dt} = (r_j^2 - \theta_j) / \tau_\theta$

**Key properties:**
- If $r_j > \theta_j$: LTP (Hebbian)
- If $r_j < \theta_j$: LTD (anti-Hebbian)
- The sliding threshold prevents runaway potentiation — if a cell is chronically active, $\theta$ rises until LTD dominates

## Your Task

Complete the CUDA kernels below to implement BCM plasticity in a network of LIF neurons.

In [ ]:
!nvidia-smi

In [ ]:
%%writefile bcm_sim.cu
#include <stdio.h>
#include <stdlib.h>
#include <math.h>
#include <cuda_runtime.h>

#define CUDA_CHECK(call) do { cudaError_t e=(call); \
    if(e!=cudaSuccess){fprintf(stderr,"CUDA %s:%d %s\n",__FILE__,__LINE__, \
    cudaGetErrorString(e));exit(1);}} while(0)

// Parameters in constant memory
__constant__ float c_dt;         // timestep (ms)
__constant__ float c_tau_m;      // membrane time constant (ms)
__constant__ float c_E_L;        // leak reversal (mV)
__constant__ float c_Rm;         // membrane resistance (MOhm)
__constant__ float c_V_th;       // threshold (mV)
__constant__ float c_V_reset;    // reset (mV)
__constant__ int   c_T_ref;      // refractory steps
__constant__ float c_tau_E;      // synaptic time constant (ms)
__constant__ float c_E_E;        // excitatory reversal (mV)
__constant__ float c_tau_r;      // rate filter time constant (ms)
__constant__ float c_tau_theta;  // BCM threshold time constant (ms)
__constant__ float c_eta;        // BCM learning rate
__constant__ float c_w_max;      // weight upper bound
__constant__ float c_w_min;      // weight lower bound

// ─────────────────────────────────────────────────────────────────────────────
// PART 1: Update smoothed firing rates and BCM thresholds
//
// r[j]  tracks a low-pass-filtered spike count:
//   r[j] <- r[j] * exp(-dt/tau_r) + fired[j] / dt
//   (adding fired[j]/dt converts a binary spike to instantaneous rate)
//
// theta[j] slides toward r[j]^2:
//   d_theta/dt = (r[j]^2 - theta[j]) / tau_theta
//   -> theta[j] += dt/tau_theta * (r[j]^2 - theta[j])
// ─────────────────────────────────────────────────────────────────────────────
__global__ void update_rates_and_theta(
    const int* fired,  // [N] 1 if neuron fired this step
    float* r,          // [N] smoothed firing rates (Hz)
    float* theta,      // [N] BCM modification thresholds
    int N
) {
    int j = blockIdx.x * blockDim.x + threadIdx.x;
    if (j >= N) return;

    // ??? Update smoothed rate r[j]
    // Hint: decay with exp(-dt/tau_r), then add fired[j] / (dt/1000.f)
    // (dividing by dt/1000 converts to Hz since dt is in ms)
    r[j] = ???;

    // ??? Update BCM threshold theta[j]
    // Hint: Euler step: theta += (dt / tau_theta) * (r^2 - theta)
    theta[j] += ???;
}

// ─────────────────────────────────────────────────────────────────────────────
// PART 2: LIF step (already complete — study this before Parts 1 & 3)
// ─────────────────────────────────────────────────────────────────────────────
__global__ void lif_step(
    float* V, float* g_E, int* ref,
    int* fired, const float* I_ext, int N
) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i >= N) return;

    g_E[i] *= expf(-c_dt / c_tau_E);
    fired[i] = 0;
    if (ref[i] > 0) { ref[i]--; V[i] = c_V_reset; return; }

    float I_syn = -g_E[i] * (V[i] - c_E_E);
    V[i] += c_dt / c_tau_m * (-(V[i] - c_E_L) + c_Rm * (I_ext[i] + I_syn));
    if (V[i] >= c_V_th) { V[i] = c_V_reset; ref[i] = c_T_ref; fired[i] = 1; }
}

// ─────────────────────────────────────────────────────────────────────────────
// PART 3: BCM weight update
//
// For each fired pre-neuron i, iterate over its outgoing synapses i→j:
//   dw = eta * r[i] * r[j] * (r[j] - theta[j])
//
// Apply soft weight bounds:
//   if dw > 0: scale by (w_max - w[k]) / w_max
//   if dw < 0: scale by (w[k] - w_min) / w_max
//
// Use atomicAdd to update weights[k] safely.
// ─────────────────────────────────────────────────────────────────────────────
__global__ void bcm_update(
    const int* fired,
    const int* row_ptr, const int* col_idx,
    float* weights,
    const float* r,      // smoothed rates
    const float* theta,  // BCM thresholds
    int N
) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i >= N || !fired[i]) return;

    for (int k = row_ptr[i]; k < row_ptr[i+1]; k++) {
        int j = col_idx[k];

        // ??? Compute dw using BCM rule
        float dw = ???;

        // ??? Apply soft weight bounds
        if (dw > 0.f) dw *= ???;
        else          dw *= ???;

        // ??? Update weight safely
        ???;
    }
}

// ─────────────────────────────────────────────────────────────────────────────
// PART 4: Spike propagation (already complete)
// ─────────────────────────────────────────────────────────────────────────────
__global__ void propagate(
    const int* fired, const int* row_ptr, const int* col_idx,
    const float* weights, float* g_E, int N
) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i >= N || !fired[i]) return;
    for (int k = row_ptr[i]; k < row_ptr[i+1]; k++)
        atomicAdd(&g_E[col_idx[k]], weights[k]);
}

// CSR builder (same as network_sim.cu)
void build_csr(int N, float p, float w_init,
               int** rp, int** ci, float** wv, int* nnz_out)
{
    srand(42);
    int* cnt=(int*)calloc(N,sizeof(int));
    for(int i=0;i<N;i++) for(int j=0;j<N;j++)
        if(i!=j && (float)rand()/RAND_MAX < p) cnt[i]++;
    *rp=(int*)malloc((N+1)*sizeof(int)); (*rp)[0]=0;
    for(int i=0;i<N;i++) (*rp)[i+1]=(*rp)[i]+cnt[i];
    int nnz=(*rp)[N]; *nnz_out=nnz;
    *ci=(int*)malloc(nnz*sizeof(int));
    *wv=(float*)malloc(nnz*sizeof(float));
    srand(42); int* pos=(int*)calloc(N,sizeof(int));
    for(int i=0;i<N;i++) for(int j=0;j<N;j++)
        if(i!=j && (float)rand()/RAND_MAX < p) {
            int k=(*rp)[i]+pos[i]++; (*ci)[k]=j; (*wv)[k]=w_init; }
    free(cnt); free(pos);
}

int main(int argc, char** argv)
{
    int   N    = (argc>1) ? atoi(argv[1]) : 200;
    float T_ms = (argc>2) ? atof(argv[2]) : 3000.f;
    float dt   = 0.1f;
    int   T    = (int)(T_ms / dt);

    // LIF params
    float tau_m=20.f,E_L=-65.f,Rm=10.f,V_th=-55.f,V_reset=-70.f;
    int T_ref=(int)(2.f/dt);
    float tau_E=5.f, E_E=0.f;

    // BCM params
    float tau_r    = 100.f;   // rate filter (ms)
    float tau_theta= 1000.f;  // threshold slide (ms)
    float eta      = 0.001f;  // learning rate
    float w_max    = 0.5f, w_min = 0.0f, w_init = 0.15f;
    float p_conn   = 0.2f;

    CUDA_CHECK(cudaMemcpyToSymbol(c_dt,       &dt,       sizeof(float)));
    CUDA_CHECK(cudaMemcpyToSymbol(c_tau_m,    &tau_m,    sizeof(float)));
    CUDA_CHECK(cudaMemcpyToSymbol(c_E_L,      &E_L,      sizeof(float)));
    CUDA_CHECK(cudaMemcpyToSymbol(c_Rm,       &Rm,       sizeof(float)));
    CUDA_CHECK(cudaMemcpyToSymbol(c_V_th,     &V_th,     sizeof(float)));
    CUDA_CHECK(cudaMemcpyToSymbol(c_V_reset,  &V_reset,  sizeof(float)));
    CUDA_CHECK(cudaMemcpyToSymbol(c_T_ref,    &T_ref,    sizeof(int)));
    CUDA_CHECK(cudaMemcpyToSymbol(c_tau_E,    &tau_E,    sizeof(float)));
    CUDA_CHECK(cudaMemcpyToSymbol(c_E_E,      &E_E,      sizeof(float)));
    CUDA_CHECK(cudaMemcpyToSymbol(c_tau_r,    &tau_r,    sizeof(float)));
    CUDA_CHECK(cudaMemcpyToSymbol(c_tau_theta,&tau_theta,sizeof(float)));
    CUDA_CHECK(cudaMemcpyToSymbol(c_eta,      &eta,      sizeof(float)));
    CUDA_CHECK(cudaMemcpyToSymbol(c_w_max,    &w_max,    sizeof(float)));
    CUDA_CHECK(cudaMemcpyToSymbol(c_w_min,    &w_min,    sizeof(float)));

    int *h_rp,*h_ci; float *h_wv; int nnz;
    build_csr(N,p_conn,w_init,&h_rp,&h_ci,&h_wv,&nnz);

    float *d_V,*d_gE,*d_Iext,*d_r,*d_theta,*d_wv;
    int   *d_ref,*d_fired,*d_rp,*d_ci;
    CUDA_CHECK(cudaMalloc(&d_V,    N*sizeof(float)));
    CUDA_CHECK(cudaMalloc(&d_gE,   N*sizeof(float)));
    CUDA_CHECK(cudaMalloc(&d_Iext, N*sizeof(float)));
    CUDA_CHECK(cudaMalloc(&d_r,    N*sizeof(float)));
    CUDA_CHECK(cudaMalloc(&d_theta,N*sizeof(float)));
    CUDA_CHECK(cudaMalloc(&d_ref,  N*sizeof(int)));
    CUDA_CHECK(cudaMalloc(&d_fired,N*sizeof(int)));
    CUDA_CHECK(cudaMalloc(&d_wv,   nnz*sizeof(float)));
    CUDA_CHECK(cudaMalloc(&d_rp,  (N+1)*sizeof(int)));
    CUDA_CHECK(cudaMalloc(&d_ci,   nnz*sizeof(int)));

    float* h_V=(float*)malloc(N*sizeof(float));
    float* h_Iext=(float*)malloc(N*sizeof(float));
    srand(7);
    for(int i=0;i<N;i++) {
        h_V[i]=E_L; h_Iext[i]=1.0f+0.8f*(float)rand()/RAND_MAX; }
    CUDA_CHECK(cudaMemcpy(d_V,   h_V,   N*sizeof(float),cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemcpy(d_Iext,h_Iext,N*sizeof(float),cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemset(d_gE,  0,N*sizeof(float)));
    CUDA_CHECK(cudaMemset(d_r,   0,N*sizeof(float)));
    CUDA_CHECK(cudaMemset(d_ref, 0,N*sizeof(int)));

    // Init theta to a small baseline
    float theta_init = 5.0f;  // Hz^2
    float* h_theta=(float*)malloc(N*sizeof(float));
    for(int i=0;i<N;i++) h_theta[i]=theta_init;
    CUDA_CHECK(cudaMemcpy(d_theta,h_theta,N*sizeof(float),cudaMemcpyHostToDevice));

    CUDA_CHECK(cudaMemcpy(d_wv,h_wv,nnz*sizeof(float),cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemcpy(d_rp,h_rp,(N+1)*sizeof(int),cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemcpy(d_ci,h_ci,nnz*sizeof(int),cudaMemcpyHostToDevice));

    int thr=256, blk=(N+thr-1)/thr;

    FILE* fsnap=fopen("bcm_weights.txt","w");
    FILE* fspikes=fopen("bcm_spikes.txt","w");
    int* h_fired=(int*)malloc(N*sizeof(int));

    int snap_every=(int)(500/dt);

    cudaEvent_t ev0,ev1; float sim_ms;
    CUDA_CHECK(cudaEventCreate(&ev0)); CUDA_CHECK(cudaEventCreate(&ev1));
    CUDA_CHECK(cudaEventRecord(ev0));

    for(int step=0;step<T;step++) {
        lif_step<<<blk,thr>>>(d_V,d_gE,d_ref,d_fired,d_Iext,N);
        propagate<<<blk,thr>>>(d_fired,d_rp,d_ci,d_wv,d_gE,N);

        // ??? Call update_rates_and_theta here
        ???;

        // ??? Call bcm_update here (only after a warmup of 500ms)
        if (step > (int)(500/dt)) ???;

        if(step%10==0) {
            CUDA_CHECK(cudaMemcpy(h_fired,d_fired,N*sizeof(int),cudaMemcpyDeviceToHost));
            float t_ms=step*dt;
            for(int i=0;i<N;i++) if(h_fired[i]) fprintf(fspikes,"%d %.1f\n",i,t_ms);
        }
        if(step%snap_every==0) {
            CUDA_CHECK(cudaMemcpy(h_wv,d_wv,nnz*sizeof(float),cudaMemcpyDeviceToHost));
            fprintf(fsnap,"# t=%.0f ms\n",step*dt);
            for(int k=0;k<nnz;k++) fprintf(fsnap,"%.6f\n",h_wv[k]);
        }
    }

    CUDA_CHECK(cudaEventRecord(ev1)); CUDA_CHECK(cudaEventSynchronize(ev1));
    CUDA_CHECK(cudaEventElapsedTime(&sim_ms,ev0,ev1));
    printf("N=%d, T=%.0f ms, GPU=%.1f ms, speedup=%.1fx\n",
           N, T_ms, sim_ms, T_ms/sim_ms);
    fclose(fsnap); fclose(fspikes);
    CUDA_CHECK(cudaEventDestroy(ev0)); CUDA_CHECK(cudaEventDestroy(ev1));
    cudaFree(d_V);cudaFree(d_gE);cudaFree(d_Iext);cudaFree(d_r);
    cudaFree(d_theta);cudaFree(d_ref);cudaFree(d_fired);
    cudaFree(d_wv);cudaFree(d_rp);cudaFree(d_ci);
    free(h_V);free(h_Iext);free(h_theta);free(h_fired);
    free(h_rp);free(h_ci);free(h_wv);
    return 0;
}

## Compile and Run

Once you've filled in the `???` placeholders, compile and run:

In [ ]:
!nvcc -O2 -o bcm_sim bcm_sim.cu -lm && ./bcm_sim 200 3000

## Visualize Results

If BCM is working correctly, you should see:
1. Weight distribution shift from uniform → bimodal (or skewed)
2. Network firing rate stabilize (homeostatic regulation via sliding threshold)
3. Neurons that receive correlated input develop stronger connections

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Load and plot weight snapshots
snapshots = {}
with open('bcm_weights.txt') as f:
    current_t, current_w = None, []
    for line in f:
        line = line.strip()
        if line.startswith('#'):
            if current_t is not None:
                snapshots[current_t] = np.array(current_w)
            current_t = float(line.split('=')[1].split()[0])
            current_w = []
        else:
            current_w.append(float(line))
    if current_t is not None:
        snapshots[current_t] = np.array(current_w)

times = sorted(snapshots.keys())
n_plots = min(len(times), 6)
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.flatten()

for idx, t in enumerate(times[:n_plots]):
    w = snapshots[t]
    axes[idx].hist(w, bins=40, color='teal', alpha=0.8, edgecolor='white')
    axes[idx].axvline(w.mean(), color='crimson', linestyle='--',
                       label=f'μ={w.mean():.3f}')
    axes[idx].set_title(f't = {t:.0f} ms', fontsize=12)
    axes[idx].set_xlabel('Weight w')
    axes[idx].set_ylabel('Count')
    axes[idx].legend(fontsize=9)
    axes[idx].grid(True, alpha=0.3)
    axes[idx].set_xlim(0, 0.5)

plt.suptitle('BCM: Weight Distribution Over Time', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## Reflection Questions

1. **Homeostasis:** How does the sliding threshold $\theta$ prevent runaway excitation? What happens if $\tau_\theta$ is very short (fast sliding)?

2. **Comparison with STDP:** BCM uses rate information; STDP uses spike timing. Which would be harder to implement efficiently on GPU for very large networks? Why?

3. **Soft bounds:** We applied soft bounds to BCM updates. What would happen without them? What distribution would weights converge to?

4. **Thread safety:** The `bcm_update` kernel uses `atomicAdd`. Could two threads write to the same weight entry? Under what connectivity structure would this happen most frequently?

## Challenge (open-ended)

Extend the simulation to include two **populations** with different input statistics:
- Population A (neurons 0–99): receives correlated Poisson input (shared noise)
- Population B (neurons 100–199): receives independent Poisson input

Observe how BCM differentially strengthens synapses within population A vs cross-population. This replicates classic BCM ocular dominance column formation experiments.